In [2]:
from pathlib import Path
import subprocess
import sys

BASE_DIR = Path(r"D:\AI_IT_HELPDESK")
APP_PATH = BASE_DIR / "frontend" / "app.py"

print("Application path:", APP_PATH)
print("File exists:", APP_PATH.exists())

if not APP_PATH.exists():
    raise FileNotFoundError(
        f"Streamlit app not found: {APP_PATH}"
    )


Application path: D:\AI_IT_HELPDESK\frontend\app.py
File exists: True


In [3]:
import subprocess
import sys

process = subprocess.Popen([
    sys.executable,
    "-m",
    "streamlit",
    "run",
    str(APP_PATH),
    "--server.port=8501"
])

print("Admin Dashboard application started!")

Admin Dashboard application started!


In [4]:
process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "streamlit",
        "run",
        str(APP_PATH),
        "--server.port=8501"
    ],
    cwd=str(APP_PATH.parent)
)

print("Streamlit started!")
print("Process ID:", process.pid)
print("URL: http://localhost:8501")


Streamlit started!
Process ID: 32820
URL: http://localhost:8501


In [ ]:
# Check password security library

In [5]:
import sys
import subprocess

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "show",
        "bcrypt"
    ],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("bcrypt is already installed.")
    print(result.stdout)
else:
    print("Installing bcrypt...")
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "bcrypt"
    ])
    print("bcrypt installed successfully!")

bcrypt is already installed.
Name: bcrypt
Version: 5.0.0
Summary: Modern password hashing for your software and your servers
Home-page: https://github.com/pyca/bcrypt/
Author: 
Author-email: The Python Cryptographic Authority developers <cryptography-dev@python.org>
License: Apache-2.0
Location: C:\Users\Nimesh\anaconda3\Lib\site-packages
Requires: 
Required-by: chromadb, spyder



In [ ]:
# Check existing users

In [6]:
import sqlite3
import pandas as pd

DB_PATH = r"D:\AI_IT_HELPDESK\database\helpdesk.db"

conn = sqlite3.connect(DB_PATH)

users_df = pd.read_sql_query(
    """
    SELECT user_id, username, email, role, created_at
    FROM users
    """,
    conn
)

display(users_df)

conn.close()

,user_id,username,email,role,created_at


In [ ]:
# Create Initial Admin Account

In [8]:
import sqlite3
import bcrypt
from pathlib import Path
from getpass import getpass

DB_PATH = Path(r"D:\AI_IT_HELPDESK\database\helpdesk.db")

# Admin details
username = input("Enter admin username: ").strip()
email = input("Enter admin email: ").strip().lower()

password = getpass("Enter admin password: ")
confirm_password = getpass("Confirm admin password: ")

if not username or not email:
    raise ValueError("Username and email are required.")

if len(password) < 12:
    raise ValueError("Use a password with at least 12 characters.")

if password != confirm_password:
    raise ValueError("Passwords do not match.")

# Generate secure password hash
password_hash = bcrypt.hashpw(
    password.encode("utf-8"),
    bcrypt.gensalt()
).decode("utf-8")

conn = sqlite3.connect(DB_PATH)

try:
    cursor = conn.cursor()

    # Check duplicate username or email
    cursor.execute(
        """
        SELECT user_id
        FROM users
        WHERE LOWER(username) = LOWER(?)
           OR LOWER(email) = LOWER(?)
        """,
        (username, email)
    )

    if cursor.fetchone():
        raise ValueError("Username or email already exists.")

    cursor.execute(
        """
        INSERT INTO users
        (username, email, password_hash, role)
        VALUES (?, ?, ?, 'admin')
        """,
        (username, email, password_hash)
    )

    admin_id = cursor.lastrowid
    conn.commit()

    print("Admin account created successfully!")
    print("Admin User ID:", admin_id)
    print("Username:", username)
    print("Role: admin")

except Exception:
    conn.rollback()
    raise

finally:
    conn.close()

# Clear password variables
password = None
confirm_password = None
password_hash = None

Enter admin username:  admin
Enter admin email:  admin@admin.com
Enter admin password:  ········
Confirm admin password:  ········


Admin account created successfully!
Admin User ID: 1
Username: admin
Role: admin


In [ ]:
# Verify Admin Account

In [9]:
import sqlite3
import pandas as pd

DB_PATH = r"D:\AI_IT_HELPDESK\database\helpdesk.db"

conn = sqlite3.connect(DB_PATH)

admin_df = pd.read_sql_query(
    """
    SELECT user_id, username, email, role, created_at
    FROM users
    WHERE role = 'admin'
    """,
    conn
)

display(admin_df)

conn.close()


,user_id,username,email,role,created_at
0,1,admin,admin@admin.com,admin,2026-11-02 09:41:52


In [ ]:
# Create login authentication function

In [10]:
import sqlite3
import bcrypt
from pathlib import Path

DB_PATH = Path(r"D:\AI_IT_HELPDESK\database\helpdesk.db")

def authenticate_user(username, password):

    conn = sqlite3.connect(DB_PATH)

    try:
        cursor = conn.cursor()

        cursor.execute(
            """
            SELECT user_id, username, password_hash, role
            FROM users
            WHERE LOWER(username) = LOWER(?)
            """,
            (username.strip(),)
        )

        user = cursor.fetchone()

        if user is None:
            return None

        user_id, db_username, password_hash, role = user

        if not password_hash:
            return None

        password_valid = bcrypt.checkpw(
            password.encode("utf-8"),
            password_hash.encode("utf-8")
        )

        if password_valid:
            return {
                "user_id": user_id,
                "username": db_username,
                "role": role
            }

        return None

    finally:
        conn.close()


print("Authentication function created successfully!")


Authentication function created successfully!


In [ ]:
# Test login authentication

In [12]:
from getpass import getpass

test_username = input("Username: ").strip()
test_password = getpass("Password: ")

login_result = authenticate_user(
    test_username,
    test_password
)

if login_result:

    print("Login successful!")
    print("User ID:", login_result["user_id"])
    print("Username:", login_result["username"])
    print("Role:", login_result["role"])

else:

    print("Invalid username or password.")

test_password = None


Username:  admin
Password:  ········


Login successful!
User ID: 1
Username: admin
Role: admin


In [ ]:
# Backup current application

In [13]:
from pathlib import Path
import shutil

BASE_DIR = Path(r"D:\AI_IT_HELPDESK")
APP_PATH = BASE_DIR / "frontend" / "app.py"

BACKUP_PATH = BASE_DIR / "frontend" / "app_backup_step13.py"

shutil.copy2(APP_PATH, BACKUP_PATH)

print("Application backup completed!")
print(BACKUP_PATH)

Application backup completed!
D:\AI_IT_HELPDESK\frontend\app_backup_step13.py


In [ ]:
# Authentication-enabled application

In [14]:
app_code = r'''
import streamlit as st
import sqlite3
import bcrypt
import joblib
import chromadb
import pandas as pd

from pathlib import Path
from sentence_transformers import SentenceTransformer

BASE_DIR = Path(r"D:\AI_IT_HELPDESK")
DB_PATH = BASE_DIR / "database" / "helpdesk.db"
MODEL_PATH = BASE_DIR / "models" / "it_helpdesk_classifier.pkl"
VECTOR_DIR = BASE_DIR / "knowledge_base" / "vector_store"

st.set_page_config(
    page_title="AI IT Helpdesk",
    page_icon="🤖",
    layout="wide"
)

# ----------------------------------
# AUTHENTICATION
# ----------------------------------

def authenticate_user(username, password):

    conn = sqlite3.connect(DB_PATH)

    try:
        cursor = conn.cursor()

        cursor.execute(
            """
            SELECT user_id, username, password_hash, role
            FROM users
            WHERE LOWER(username) = LOWER(?)
            """,
            (username.strip(),)
        )

        user = cursor.fetchone()

        if not user or not user[2]:
            return None

        valid = bcrypt.checkpw(
            password.encode("utf-8"),
            user[2].encode("utf-8")
        )

        if valid:
            return {
                "user_id": user[0],
                "username": user[1],
                "role": user[3]
            }

        return None

    finally:
        conn.close()


def login_required():
    return "user" in st.session_state


# ----------------------------------
# LOGIN PAGE
# ----------------------------------

if not login_required():

    st.title("🤖 AI-Based IT Helpdesk")
    st.subheader("Secure Login")

    with st.form("login_form"):

        username = st.text_input("Username")
        password = st.text_input(
            "Password",
            type="password"
        )

        login_button = st.form_submit_button(
            "Login",
            type="primary"
        )

        if login_button:

            if not username.strip() or not password:
                st.error("Enter username and password.")

            else:

                user = authenticate_user(
                    username,
                    password
                )

                if user:

                    st.session_state["user"] = user
                    st.session_state["analysis"] = None

                    st.success("Login successful!")
                    st.rerun()

                else:

                    st.error("Invalid username or password.")

    st.stop()


# ----------------------------------
# CURRENT USER
# ----------------------------------

current_user = st.session_state["user"]

st.sidebar.title("Helpdesk Menu")

st.sidebar.write(
    f"Welcome, {current_user['username']}"
)

st.sidebar.caption(
    f"Role: {current_user['role']}"
)

if st.sidebar.button("Logout"):

    for key in ["user", "analysis"]:
        st.session_state.pop(key, None)

    st.rerun()


# ----------------------------------
# ROLE-BASED NAVIGATION
# ----------------------------------

if current_user["role"] == "admin":

    menu_options = [
        "AI Helpdesk",
        "My Tickets",
        "Admin Dashboard"
    ]

else:

    menu_options = [
        "AI Helpdesk",
        "My Tickets"
    ]

menu = st.sidebar.radio(
    "Navigation",
    menu_options
)

st.title("🤖 AI-Based IT Helpdesk")


# ----------------------------------
# LOAD AI COMPONENTS
# ----------------------------------

@st.cache_resource
def load_ai_components():

    classifier = joblib.load(MODEL_PATH)

    client = chromadb.PersistentClient(
        path=str(VECTOR_DIR)
    )

    collection = client.get_collection(
        name="it_helpdesk_knowledge"
    )

    embedding_model = SentenceTransformer(
        "sentence-transformers/all-MiniLM-L6-v2",
        device="cpu"
    )

    return classifier, collection, embedding_model


classifier, collection, embedding_model = load_ai_components()


# ----------------------------------
# AI FUNCTIONS
# ----------------------------------

def classify_issue(query):

    category = classifier.predict([query])[0]

    confidence = None

    if hasattr(classifier, "predict_proba"):

        probabilities = classifier.predict_proba([query])[0]

        confidence = float(max(probabilities))

    return category, confidence


def retrieve_knowledge(query, category, top_k=3):

    embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    ).tolist()

    results = collection.query(
        query_embeddings=embedding,
        n_results=top_k,
        where={"category": category}
    )

    return (
        results.get("documents", [[]])[0],
        results.get("metadatas", [[]])[0]
    )


# ----------------------------------
# CREATE TICKET
# ----------------------------------

def create_support_ticket(
    title,
    description,
    category_name,
    priority,
    user_id
):

    conn = sqlite3.connect(DB_PATH)

    try:

        cursor = conn.cursor()

        cursor.execute(
            """
            SELECT category_id
            FROM categories
            WHERE LOWER(category_name) = LOWER(?)
            """,
            (category_name,)
        )

        category = cursor.fetchone()

        if category is None:
            raise ValueError("Invalid ticket category.")

        cursor.execute(
            """
            INSERT INTO tickets
            (user_id, title, description, category_id, priority, status)
            VALUES (?, ?, ?, ?, ?, 'Open')
            """,
            (
                user_id,
                title,
                description,
                category[0],
                priority
            )
        )

        ticket_id = cursor.lastrowid

        cursor.execute(
            """
            INSERT INTO audit_logs
            (user_id, ticket_id, action, details)
            VALUES (?, ?, ?, ?)
            """,
            (
                user_id,
                ticket_id,
                "TICKET_CREATED",
                f"Created by {current_user['username']}: {title}"
            )
        )

        conn.commit()

        return ticket_id

    except Exception:

        conn.rollback()
        raise

    finally:

        conn.close()


# ----------------------------------
# AI HELPDESK
# ----------------------------------

if menu == "AI Helpdesk":

    st.subheader("How can we help you?")

    query = st.text_area(
        "Describe your IT issue:",
        height=120
    )

    if st.button("Analyse Issue", type="primary"):

        if not query.strip():

            st.warning("Enter your issue.")

        else:

            with st.spinner("AI analysing issue..."):

                category, confidence = classify_issue(query)

                documents, metadatas = retrieve_knowledge(
                    query,
                    category
                )

            st.session_state["analysis"] = {
                "query": query,
                "category": category,
                "confidence": confidence,
                "documents": documents,
                "metadatas": metadatas
            }

    analysis = st.session_state.get("analysis")

    if analysis:

        st.success("Issue analysis completed!")

        col1, col2 = st.columns(2)

        col1.metric(
            "Predicted Category",
            analysis["category"]
        )

        if analysis["confidence"] is not None:

            col2.metric(
                "Classification Confidence",
                f"{analysis['confidence']:.2%}"
            )

        st.subheader("Troubleshooting Guidance")

        for i, document in enumerate(analysis["documents"]):

            title = analysis["metadatas"][i].get(
                "title",
                f"Article {i+1}"
            )

            with st.expander(title):
                st.write(document)

        st.subheader("Create Support Ticket")

        with st.form("ticket_form"):

            title = st.text_input(
                "Ticket Title",
                value=analysis["query"][:100]
            )

            description = st.text_area(
                "Description",
                value=analysis["query"]
            )

            categories = [
                "Account & Password",
                "Network & VPN",
                "Printer",
                "Software",
                "System Health"
            ]

            predicted = analysis["category"]

            category_index = (
                categories.index(predicted)
                if predicted in categories else 0
            )

            category = st.selectbox(
                "Category",
                categories,
                index=category_index
            )

            priority = st.selectbox(
                "Priority",
                ["Low", "Medium", "High"],
                index=1
            )

            submit = st.form_submit_button(
                "Create Support Ticket"
            )

            if submit:

                if not title.strip() or not description.strip():

                    st.error("Complete all required fields.")

                else:

                    ticket_id = create_support_ticket(
                        title.strip(),
                        description.strip(),
                        category,
                        priority,
                        current_user["user_id"]
                    )

                    st.success(
                        f"Ticket created successfully! ID: {ticket_id}"
                    )


# ----------------------------------
# MY TICKETS
# ----------------------------------

elif menu == "My Tickets":

    st.subheader("🎫 My Support Tickets")

    conn = sqlite3.connect(DB_PATH)

    if current_user["role"] == "admin":

        query = """
            SELECT t.ticket_id, t.title, c.category_name,
                   t.priority, t.status, t.created_at
            FROM tickets t
            LEFT JOIN categories c
            ON t.category_id = c.category_id
            ORDER BY t.ticket_id DESC
        """

        params = ()

    else:

        query = """
            SELECT t.ticket_id, t.title, c.category_name,
                   t.priority, t.status, t.created_at
            FROM tickets t
            LEFT JOIN categories c
            ON t.category_id = c.category_id
            WHERE t.user_id = ?
            ORDER BY t.ticket_id DESC
        """

        params = (current_user["user_id"],)

    tickets = pd.read_sql_query(
        query,
        conn,
        params=params
    )

    conn.close()

    st.dataframe(
        tickets,
        use_container_width=True,
        hide_index=True
    )


# ----------------------------------
# ADMIN DASHBOARD
# ----------------------------------

elif menu == "Admin Dashboard":

    # Server-side role authorization
    if current_user["role"] != "admin":
        st.error("Access denied.")
        st.stop()

    st.subheader("📊 IT Support Admin Dashboard")

    conn = sqlite3.connect(DB_PATH)

    total = conn.execute(
        "SELECT COUNT(*) FROM tickets"
    ).fetchone()[0]

    open_count = conn.execute(
        "SELECT COUNT(*) FROM tickets WHERE status='Open'"
    ).fetchone()[0]

    progress_count = conn.execute(
        "SELECT COUNT(*) FROM tickets WHERE status='In Progress'"
    ).fetchone()[0]

    resolved_count = conn.execute(
        "SELECT COUNT(*) FROM tickets WHERE status='Resolved'"
    ).fetchone()[0]

    high_count = conn.execute(
        "SELECT COUNT(*) FROM tickets WHERE priority='High'"
    ).fetchone()[0]

    c1, c2, c3 = st.columns(3)

    c1.metric("Total Tickets", total)
    c2.metric("Open Tickets", open_count)
    c3.metric("In Progress", progress_count)

    c4, c5 = st.columns(2)

    c4.metric("Resolved", resolved_count)
    c5.metric("High Priority", high_count)

    st.divider()

    tickets = pd.read_sql_query(
        """
        SELECT t.ticket_id, t.title, t.description,
               c.category_name, t.priority,
               t.status, t.created_at
        FROM tickets t
        LEFT JOIN categories c
        ON t.category_id = c.category_id
        ORDER BY t.ticket_id DESC
        """,
        conn
    )

    conn.close()

    st.subheader("Ticket Management")

    st.dataframe(
        tickets,
        use_container_width=True,
        hide_index=True
    )

    if not tickets.empty:

        selected_id = st.selectbox(
            "Select Ticket",
            tickets["ticket_id"].tolist()
        )

        row = tickets[
            tickets["ticket_id"] == selected_id
        ].iloc[0]

        with st.form("admin_update"):

            status = st.selectbox(
                "Status",
                ["Open", "In Progress", "Resolved"],
                index=["Open", "In Progress", "Resolved"].index(row["status"])
            )

            priority = st.selectbox(
                "Priority",
                ["Low", "Medium", "High"],
                index=["Low", "Medium", "High"].index(row["priority"])
            )

            save = st.form_submit_button("Save Changes")

            if save:

                conn = sqlite3.connect(DB_PATH)

                conn.execute(
                    """
                    UPDATE tickets
                    SET status=?,
                        priority=?,
                        updated_at=CURRENT_TIMESTAMP
                    WHERE ticket_id=?
                    """,
                    (status, priority, int(selected_id))
                )

                conn.execute(
                    """
                    INSERT INTO audit_logs
                    (user_id, ticket_id, action, details)
                    VALUES (?, ?, ?, ?)
                    """,
                    (
                        current_user["user_id"],
                        int(selected_id),
                        "TICKET_UPDATED",
                        f"Status: {status}; Priority: {priority}"
                    )
                )

                conn.commit()
                conn.close()

                st.success("Ticket updated successfully!")
                st.rerun()


st.divider()

st.caption(
    "MSc Computer Science | AI-Based IT Helpdesk"
)
'''

APP_PATH.write_text(app_code, encoding="utf-8")

print("Authentication-enabled application created!")
print("File:", APP_PATH)

Authentication-enabled application created!
File: D:\AI_IT_HELPDESK\frontend\app.py


In [15]:
import subprocess
import sys

process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "streamlit",
        "run",
        str(APP_PATH),
        "--server.port=8501"
    ],
    cwd=str(APP_PATH.parent)
)

print("Authentication-enabled application started.")
print("http://localhost:8501")

Authentication-enabled application started.
http://localhost:8501


In [ ]:
# Create Normal User Account

In [16]:
import sqlite3
import bcrypt
from pathlib import Path
from getpass import getpass

DB_PATH = Path(r"D:\AI_IT_HELPDESK\database\helpdesk.db")

username = input("Enter new username: ").strip()
email = input("Enter email: ").strip().lower()

password = getpass("Enter password: ")
confirm_password = getpass("Confirm password: ")

if not username or not email:
    raise ValueError("Username and email are required.")

if len(password) < 12:
    raise ValueError("Password must contain at least 12 characters.")

if password != confirm_password:
    raise ValueError("Passwords do not match.")

password_hash = bcrypt.hashpw(
    password.encode("utf-8"),
    bcrypt.gensalt()
).decode("utf-8")

conn = sqlite3.connect(DB_PATH)

try:
    cursor = conn.cursor()

    cursor.execute(
        """
        SELECT user_id FROM users
        WHERE LOWER(username) = LOWER(?)
           OR LOWER(email) = LOWER(?)
        """,
        (username, email)
    )

    if cursor.fetchone():
        raise ValueError("Username or email already exists.")

    cursor.execute(
        """
        INSERT INTO users
        (username, email, password_hash, role)
        VALUES (?, ?, ?, 'user')
        """,
        (username, email, password_hash)
    )

    conn.commit()

    print("Normal user created successfully!")
    print("Username:", username)
    print("Role: user")

except Exception:
    conn.rollback()
    raise

finally:
    conn.close()
    password = None
    confirm_password = None
    password_hash = None


Enter new username:  user
Enter email:  user@admin.com
Enter password:  ········
Confirm password:  ········


Normal user created successfully!
Username: user
Role: user


In [ ]:
# Verify ticket ownership in SQLite

In [17]:
import sqlite3
import pandas as pd

DB_PATH = r"D:\AI_IT_HELPDESK\database\helpdesk.db"

conn = sqlite3.connect(DB_PATH)

ticket_ownership_df = pd.read_sql_query(
    """
    SELECT
        t.ticket_id,
        t.user_id,
        u.username,
        t.title,
        c.category_name,
        t.priority,
        t.status,
        t.created_at
    FROM tickets t
    LEFT JOIN users u
        ON t.user_id = u.user_id
    LEFT JOIN categories c
        ON t.category_id = c.category_id
    ORDER BY t.ticket_id DESC
    """,
    conn
)

display(ticket_ownership_df)

conn.close()


,ticket_id,user_id,username,title,category_name,priority,status,created_at
0,2,None,None,printer not working,Printer,Low,Open,2026-11-02 09:22:50
1,1,None,None,Printer is offline,Printer,Medium,Open,2026-11-02 09:12:43


In [ ]:
# Verify new ticket ownership

In [20]:
import sqlite3
import pandas as pd

DB_PATH = r"D:\AI_IT_HELPDESK\database\helpdesk.db"

conn = sqlite3.connect(DB_PATH)

result = pd.read_sql_query(
    """
    SELECT
        t.ticket_id,
        t.user_id,
        u.username,
        t.title,
        t.status
    FROM tickets t
    LEFT JOIN users u
        ON t.user_id = u.user_id
    ORDER BY t.ticket_id DESC
    LIMIT 5
    """,
    conn
)

display(result)

conn.close()


,ticket_id,user_id,username,title,status
0,4,1.0,admin,scanner not working,Open
1,3,2.0,user,my account not working,Open
2,2,NaN,NaN,printer not working,Open
3,1,NaN,NaN,Printer is offline,Open


In [ ]:
# Verify User Ticket Isolation

In [21]:
import sqlite3
import pandas as pd

DB_PATH = r"D:\AI_IT_HELPDESK\database\helpdesk.db"

conn = sqlite3.connect(DB_PATH)

user_tickets = pd.read_sql_query(
    """
    SELECT
        ticket_id,
        user_id,
        title,
        status,
        priority
    FROM tickets
    WHERE user_id = 2
    ORDER BY ticket_id DESC
    """,
    conn
)

display(user_tickets)

conn.close()


,ticket_id,user_id,title,status,priority
0,3,2,my account not working,Open,High


In [ ]:
# Verify Admin Ticket Access

In [22]:
conn = sqlite3.connect(DB_PATH)

admin_view = pd.read_sql_query(
    """
    SELECT
        ticket_id,
        user_id,
        title,
        status,
        priority
    FROM tickets
    ORDER BY ticket_id DESC
    """,
    conn
)

display(admin_view)

conn.close()


,ticket_id,user_id,title,status,priority
0,4,1.0,scanner not working,Open,Medium
1,3,2.0,my account not working,Open,High
2,2,NaN,printer not working,Open,Low
3,1,NaN,Printer is offline,Open,Medium


In [ ]:
# Audit Log Verification

In [23]:

conn = sqlite3.connect(DB_PATH)

audit_df = pd.read_sql_query(
    """
    SELECT
        log_id,
        user_id,
        ticket_id,
        action,
        details,
        created_at
    FROM audit_logs
    ORDER BY log_id DESC
    LIMIT 20
    """,
    conn
)

display(audit_df)


,log_id,user_id,ticket_id,action,details,created_at
0,4,1.0,4,TICKET_CREATED,Created by admin: scanner not working,2026-11-02 10:15:57
1,3,2.0,3,TICKET_CREATED,Created by user: my account not working,2026-11-02 10:14:56
2,2,NaN,2,TICKET_CREATED,Created from AI Helpdesk: printer not working,2026-11-02 09:22:50
3,1,NaN,1,TICKET_CREATED,Ticket created: Printer is offline,2026-11-02 09:12:43
